In [1]:
import datetime as dt
import s3fs
from co2sat.data.goes16 import (
    extract_day_matrix,
    find_scan_for_hour,
    extract_bands_at_point,
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
fs = s3fs.S3FileSystem(anon=True)

In [3]:
m = extract_day_matrix(fs, dt.date(2021, 4, 1), lon=-95.63, lat=29.49)  # W A Parish

In [4]:
print(f"Filled: {(~__import__('numpy').isnan(m)).sum()}")

Filled: 384


In [ ]:
target = dt.datetime(2021, 4, 1, 18, 0)
path = find_scan_for_hour(fs, target)
print("Path found:", path)

In [ ]:
values = extract_bands_at_point(fs, path, lon=-95.63, lat=29.49)
print(values)

In [5]:
hours = list(range(24))
refl_bands = [f"C{b:02d}" for b in range(1, 7)]  # bands 1-6
therm_bands = [f"C{b:02d}" for b in range(7, 17)]  # bands 7-16

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        "Bands 1-6 — reflectance factor (0-1)",
        "Bands 7-16 — brightness temperature (K)",
    ),
    vertical_spacing=0.15,
)

fig.add_trace(
    go.Heatmap(
        z=m[0:6, :],
        x=hours,
        y=refl_bands,
        colorscale="Viridis",
        colorbar=dict(title="refl.", len=0.4, y=0.8),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=m[6:16, :],
        x=hours,
        y=therm_bands,
        colorscale="Inferno",
        colorbar=dict(title="K", len=0.4, y=0.2),
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text="Hour (UTC)", row=2, col=1)
fig.update_layout(
    height=650,
    width=950,
    title_text="W. A. Parish — 2021-04-01 — 16 bands × 24 hours",
)
fig.show()